# Colab GPU Bootstrap — video-action-mlops

**Mục đích:** dùng GPU T4 miễn phí của Colab làm "worker" chạy các stage cần GPU
trong `dvc.yaml` (train_phase1, extract_features, train_phase2), theo **Phương
án A** (thủ công/bán tự động) đã chốt ở phiên 6.3.

**Ghi chú tuân thủ chính sách Colab (quan trọng):** notebook này CHỈ chạy khi
bạn đang chủ động ngồi tương tác (interactive compute) — KHÔNG để chạy nền
không giám sát, KHÔNG dùng làm self-hosted CI runner (đã phân tích và loại bỏ
phương án đó ở phiên 6.3, xem `docs/decisions/0002-gpu-strategy-colab-manual.md`
sẽ viết ở tuần 12). Chạy xong nhớ **Runtime > Disconnect and delete runtime**
để trả tài nguyên cho người dùng khác.

**Trước khi chạy:** vào biểu tượng chìa khoá 🔑 bên trái ("Secrets"), thêm các
secret sau (bật "Notebook access" cho từng cái):
- `DAGSHUB_USERNAME`, `DAGSHUB_TOKEN` (đã tạo ở phiên 4.1)
- `MLFLOW_TRACKING_URI` (dạng `https://dagshub.com/<user>/<repo>.mlflow`)
- `GIT_REPO_URL` (dạng `https://github.com/<user>/<repo>.git`)
- `GITHUB_PAT` — Personal Access Token **fine-grained**, chỉ scope đúng 1 repo
  này, quyền `Contents: Read and write` — KHÔNG dùng classic token full quyền
- `GIT_USER_NAME`, `GIT_USER_EMAIL` — dùng để `git commit` có danh tính hợp lệ


## 1. Kiểm tra GPU thật trước khi làm gì khác

In [ ]:
!nvidia-smi
import torch
print("torch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))


## 1.5. Mount Google Drive — lưu trạng thái BỀN VỮNG qua các lần disconnect (phiên 13.3)

`/content` là đĩa TẠM của máy ảo Colab — mất sạch khi session kết thúc (hết
quota GPU, bị Google chủ động ngắt, đóng tab...). Free tier T4 hay bị ngắt
giữa chừng, nên 2 thứ sau CẦN sống sót qua ranh giới session:

1. **Resume state** (phiên 13.2) — không lưu bền vững thì mỗi lần bị ngắt
   giữa chừng lại phải train lại từ epoch 0, tốn quota GPU vô ích.
2. **Cache video raw** (`data/raw/`) — UCF101/UCF11 (phiên 13.1) khá nặng,
   tải lại từ DagsHub remote (`dvc pull`) mỗi phiên Colab mới vừa chậm vừa
   tốn băng thông free-tier remote storage.

Google Drive cá nhân free (15GB) đủ cho cả 2 việc này.


In [ ]:
import os
from pathlib import Path

from google.colab import drive

drive.mount("/content/drive")

# Thư mục BỀN VỮNG trên Drive — sống sót qua mọi lần disconnect (khác
# /content, ephemeral). resume_state (phiên 13.2) và raw_cache dùng chung
# 1 gốc để dễ quản lý/dọn tay khi cần (vd hết dung lượng Drive).
DRIVE_STATE_DIR = Path("/content/drive/MyDrive/video-action-mlops-state")
RESUME_DIR = DRIVE_STATE_DIR / "resume"
RAW_CACHE_DIR = DRIVE_STATE_DIR / "raw_cache"
RESUME_DIR.mkdir(parents=True, exist_ok=True)
RAW_CACHE_DIR.mkdir(parents=True, exist_ok=True)

# scripts/run_train_phase1.py và run_train_phase2.py (phiên 13.2) tự đọc
# biến môi trường này làm mặc định cho --resume-state — chỉ cần set Ở
# ĐÂY, không cần sửa dvc.yaml hay truyền tay --resume-state mỗi lần.
os.environ["RESUME_DIR"] = str(RESUME_DIR)

print("RESUME_DIR    ->", os.environ["RESUME_DIR"])
print("RAW_CACHE_DIR ->", RAW_CACHE_DIR)


## 2. Đọc secret từ Colab, KHÔNG hardcode vào cell

Dùng `google.colab.userdata` — secret được Google mã hoá riêng cho notebook
này, không hiện trong output, không bị lưu vào lịch sử `.ipynb` khi bạn share
file (khác hẳn việc gõ thẳng token vào 1 cell).

In [ ]:
from google.colab import userdata

REQUIRED_SECRETS = [
    "DAGSHUB_USERNAME", "DAGSHUB_TOKEN", "MLFLOW_TRACKING_URI",
    "GIT_REPO_URL", "GITHUB_PAT", "GIT_USER_NAME", "GIT_USER_EMAIL",
]

secrets = {}
missing = []
for key in REQUIRED_SECRETS:
    try:
        secrets[key] = userdata.get(key)
    except Exception:
        missing.append(key)

if missing:
    raise RuntimeError(
        f"Thiếu secret trong Colab: {missing} — vào icon chìa khoá bên trái, "
        f"thêm đủ, bật 'Notebook access', rồi chạy lại cell này."
    )

print("Đã đọc đủ", len(secrets), "secret. Không in giá trị thật ra output.")


## 3. Clone (hoặc pull nếu đã clone từ phiên trước trong cùng session) repo

In [ ]:
import os
import subprocess

REPO_DIR = "/content/video-action-mlops"

if not os.path.isdir(REPO_DIR):
    subprocess.run(["git", "clone", secrets["GIT_REPO_URL"], REPO_DIR], check=True)
else:
    print("Repo đã tồn tại trong session này, pull thay vì clone lại.")
    subprocess.run(["git", "pull"], cwd=REPO_DIR, check=True)

os.chdir(REPO_DIR)
subprocess.run(["git", "config", "user.name", secrets["GIT_USER_NAME"]], check=True)
subprocess.run(["git", "config", "user.email", secrets["GIT_USER_EMAIL"]], check=True)
print("cwd hiện tại:", os.getcwd())


## 4. Ghi file `.env` thật (đọc bởi `config/loader.py`, phiên 1.2)

`load_config()` gọi `load_dotenv()` tự tìm `.env` ở thư mục hiện tại — chỉ cần
file này tồn tại trên đĩa Colab (ephemeral, mất khi session kết thúc), KHÔNG
commit vào git (đã có trong `.gitignore` từ phiên 1.3).

In [ ]:
env_content = f"""DAGSHUB_USERNAME={secrets['DAGSHUB_USERNAME']}
DAGSHUB_TOKEN={secrets['DAGSHUB_TOKEN']}
MLFLOW_TRACKING_URI={secrets['MLFLOW_TRACKING_URI']}
"""

with open(".env", "w") as f:
    f.write(env_content)

print(".env đã ghi (nội dung không in ra để tránh lộ token trong output cell).")


## 5. Cài dependency — CỐ TÌNH bỏ qua torch/torchvision

Colab đã cài sẵn `torch`/`torchvision` bản khớp đúng driver CUDA của máy ảo.
Nếu để `pip install -e .` cài lại theo `pyproject.toml` (phiên 1.1/3.1), có thể
kéo về bản CPU-only hoặc lệch version CUDA — **phá GPU support** mà không báo
lỗi rõ ràng (lỗi kiểu "CUDA not available" rất khó debug nếu không biết nguyên
nhân). Dùng `--no-deps` rồi tự cài đúng danh sách cần thiết, KHÔNG đụng
torch/torchvision.

In [ ]:
# Pin CHÍNH XÁC (phiên 13.3), không chỉ ">=" như trước — tránh pip tự ý
# resolve bản MỚI HƠN trong tương lai có thể phát sinh lỗi không lường
# trước (">=" trong pyproject.toml chỉ đảm bảo TỐI THIỂU, không đảm bảo
# TƯƠNG THÍCH). Version dưới đây đã VERIFY THẬT (cài thật + chạy
# tests/unit/ thật trong sandbox, không giả lập) tại thời điểm phiên
# 13.3. LƯU Ý: verify được là verify LOCAL (unit test), CHƯA verify
# end-to-end với DagsHub thật (sandbox không có quyền truy cập mạng vào
# dagshub.com) — nếu chạy thật trên Colab mà lỗi tương thích tracking
# server, đây là nghi phạm đầu tiên.
#
# CẬP NHẬT (phiên 13.4 đã xong, nhưng KHÔNG thay cell này bằng
# `pip install -r requirements/tracking.lock` được): tracking.lock được
# compile bao gồm CẢ torch/torchvision (base dependency trong
# pyproject.toml, mọi extras đều kế thừa — xem header file đó) — cài lại
# sẽ GHI ĐÈ torch bản Colab đã cài sẵn (đúng bản CUDA khớp GPU của máy
# ảo Colab), đúng thứ dòng `--no-deps` ở trên đang cố tránh. Danh sách
# pin tay dưới đây (không có torch) mới là lựa chọn ĐÚNG cho Colab, giữ
# nguyên, không phải nợ kỹ thuật chờ dọn.
!pip install -e . --no-deps --quiet
!pip install --quiet \
    "pydantic==2.13.5" "python-dotenv==1.2.2" "pyyaml==6.0.3" \
    "dvc[s3]==3.67.1" "mlflow==3.16.0"

import torch
assert torch.cuda.is_available(), "CUDA vừa bị mất sau khi cài deps — dừng lại, kiểm tra lại bước 5"
print("OK: torch vẫn thấy CUDA sau khi cài dependency.")


## 5.5. Khôi phục cache video raw từ Drive (phiên 13.3, sửa lại sau
khi bị phát hiện vi phạm quy ước "logic sống ở src/, notebook chỉ gọi")

`data/raw/` bị `.gitignore` theo extension (phiên 13.1) nên sau `git
clone`/`git pull` ở bước 3, thư mục này RỖNG (chỉ còn `labels.csv`). Nếu
phiên Colab TRƯỚC đã tải video về rồi, khôi phục THẲNG từ Drive cache ở
đây — nhanh hơn nhiều so với đợi `dvc pull` (bước 7) kéo lại từ remote
DagsHub.

Đặt SAU bước 5 (cài dependency) — CẦN package `video_action_mlops` đã
cài xong mới `import` được hàm dùng chung với
`tests/unit/test_raw_cache.py` (logic thật nằm ở
`data/raw_cache.py`, không viết lại trong notebook).

Nếu đây là lần chạy ĐẦU TIÊN (Drive cache còn rỗng), không sao — bước 7
(`dvc pull`) vẫn chạy bình thường, và cache sẽ được lưu lại cho các
phiên sau (xem cell ngay sau `dvc pull`).


In [ ]:
from video_action_mlops.data.raw_cache import restore_raw_from_cache

# cwd đã đổi thành REPO_DIR ở cell bước 3 -> dùng đường dẫn tương đối.
local_raw = Path("data/raw")

if restore_raw_from_cache(RAW_CACHE_DIR, local_raw):
    print(f"Tìm thấy cache video trên Drive ({RAW_CACHE_DIR}) -> đã khôi phục vào {local_raw}.")
    print("`dvc pull` ở bước 7 sẽ không phải tải lại phần này.")
else:
    print("Drive cache chưa có video nào (lần chạy đầu tiên) -> sẽ dựa vào `dvc pull` ở bước 7.")


## 6. DVC remote — trỏ credential (không commit `.dvc/config.local`, đúng thiết kế phiên 4.1)

In [ ]:
!dvc remote modify origin --local access_key_id "{secrets['DAGSHUB_TOKEN']}"
!dvc remote modify origin --local secret_access_key "{secrets['DAGSHUB_TOKEN']}"
!dvc remote list


## 7. `dvc pull` (best-effort — lần đầu có thể không có gì để pull, không sao)

In [ ]:
!dvc pull || echo "dvc pull không có gì mới hoặc lần đầu chạy — bỏ qua, tiếp tục."


## 7.5. Đồng bộ cache video raw NGƯỢC lại Drive (phiên 13.3)

Nếu `dvc pull` ở bước 7 vừa tải video MỚI về (lần đầu chạy, hoặc dataset
được cập nhật), lưu lại vào Drive cache NGAY — để phiên Colab TIẾP THEO
không phải tải lại từ remote nữa (xem lý do đầy đủ ở mục 3.5).


In [ ]:
from video_action_mlops.data.raw_cache import sync_raw_to_cache

if sync_raw_to_cache(local_raw, RAW_CACHE_DIR):
    print(f"Đã đồng bộ {local_raw} -> Drive cache ({RAW_CACHE_DIR}) cho phiên sau.")
else:
    print("Không có video nào ở data/raw sau dvc pull -- không có gì để cache.")


## 8. `dvc repro` — bước chính, cần GPU thật

Mặc định chạy TOÀN BỘ pipeline (mọi stage đang outdated). Muốn chạy 1 stage cụ
thể (vd chỉ `train_phase1` lúc đang debug), sửa dòng dưới thành
`!dvc repro train_phase1`.

In [ ]:
!dvc repro


## 9. `dvc push` — đẩy output mới (checkpoint, cache) lên DagsHub remote

In [ ]:
!dvc push


## 10. Commit + push phần Git (chỉ `dvc.lock` và các file `.dvc` con trỏ — KHÔNG phải data/checkpoint thật, những thứ đó đã nằm ở bước 9)

Dùng credential helper tạm thời (chỉ tồn tại trong lệnh này) thay vì ghi PAT
vào `.git/config` — token không bị lưu lại trên đĩa dưới dạng plaintext lâu
dài.

In [ ]:
!git add dvc.lock configs/*.yaml
!git commit -m "chore: dvc repro on Colab T4 ($(date -u +%Y-%m-%dT%H:%M:%SZ))" || echo "Không có gì thay đổi để commit."

import subprocess
push_cmd = (
    f'git -c credential.helper="!f() {{ echo username=x-access-token; '
    f'echo password={secrets["GITHUB_PAT"]}; }}; f" push'
)
subprocess.run(push_cmd, shell=True, check=True)
print("Đã push (nếu có commit mới).")


## 11. Dọn dẹp — LUÔN chạy cell này trước khi rời notebook

Xoá `.env` khỏi đĩa Colab (dù session sẽ tự mất khi ngắt kết nối, xoá tay vẫn
là thói quen tốt), và nhắc ngắt runtime để trả tài nguyên GPU cho người khác
(đúng tinh thần chính sách Colab đã nói ở đầu notebook).

In [ ]:
import os
if os.path.exists(".env"):
    os.remove(".env")
print("Đã xoá .env. Giờ vào Runtime > Disconnect and delete runtime.")
